# Fourier transform demo

This notebook demonstrates the solver-side projection and Fourier helpers:

- `solver.from_grid_vector(...)`
- `solver.to_grid_vector(...)`
- `solver.fourier(...)`

It uses the same analytic Gaussian-family cases as the unit tests and then shows a few arbitrary numerical profiles for fun.

In [1]:
from __future__ import annotations

import math

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import lax as lm

In [2]:
def legendre_solver(l: int) -> lm.boundary._types.Solver:
    mesh, _ = lm.meshes.build_mesh("legendre", "x", n=24, scale=18.0, operators={"T+L"})
    return lm.compile(
        mesh=lm.MeshSpec("legendre", "x", n=24, scale=18.0),
        channels=(lm.ChannelSpec(l=l, threshold=0.0, mass_factor=2.0),),
        solvers=(),
        grid=mesh.radii,
        momenta=jnp.linspace(0.0, 2.0, 21),
    )


def relative_error(numerical: np.ndarray, analytic: np.ndarray) -> float:
    return float(np.linalg.norm(numerical - analytic) / np.linalg.norm(analytic))


## Analytic Gaussian-family Legendre examples

In [3]:
beta = 0.5
analytic_cases = [
    (
        "Gaussian l=0",
        0,
        lambda r: r**2 * np.exp(-beta * r**2),
        lambda k: np.exp(-(k**2) / (4.0 * beta)) / (2.0 * beta) ** 1.5,
    ),
    (
        "Gaussian x polynomial l=0",
        0,
        lambda r: r**2 * (3.0 / (2.0 * beta) - r**2) * np.exp(-beta * r**2),
        lambda k: ((k**2) / (4.0 * beta**2)) * np.exp(-(k**2) / (4.0 * beta)) / (2.0 * beta) ** 1.5,
    ),
    (
        "Gaussian l=1",
        1,
        lambda r: r**3 * np.exp(-beta * r**2),
        lambda k: k * np.exp(-(k**2) / (4.0 * beta)) / (2.0 * beta) ** 2.5,
    ),
]

fig, axes = plt.subplots(len(analytic_cases), 2, figsize=(12, 10))

for row, (name, l, profile, transform) in enumerate(analytic_cases):
    solver = legendre_solver(l)
    coeffs = solver.from_grid_vector(lambda r: jnp.asarray(profile(np.asarray(r))))
    grid_values = np.asarray(solver.to_grid_vector(coeffs))
    momentum_values = np.asarray(solver.fourier(coeffs))
    grid_r = np.asarray(solver.transforms.grid_r)
    momenta = np.asarray(solver.transforms.momenta)
    expected_grid = profile(grid_r)
    expected_k = transform(momenta)
    print(name)
    print("  grid relative error    =", relative_error(grid_values, expected_grid))
    print("  momentum relative error=", relative_error(momentum_values, expected_k))
    axes[row, 0].plot(grid_r, expected_grid, label="analytic", linewidth=2.2)
    axes[row, 0].plot(grid_r, grid_values, "--", label="mesh", linewidth=1.8)
    axes[row, 0].set_title(f"{name} in r-space")
    axes[row, 0].set_xlabel("r")
    axes[row, 0].set_ylabel("u(r)")
    axes[row, 0].legend()
    axes[row, 1].plot(momenta, expected_k, label="analytic", linewidth=2.2)
    axes[row, 1].plot(momenta, momentum_values, "--", label="mesh", linewidth=1.8)
    axes[row, 1].set_title(f"{name} in k-space")
    axes[row, 1].set_xlabel("k")
    axes[row, 1].set_ylabel(r"$\tilde u(k)$")
    axes[row, 1].legend()

fig.tight_layout()


Gaussian l=0
  grid relative error    = 5.215095343970679e-16
  momentum relative error= 5.316561770249139e-07
Gaussian x polynomial l=0
  grid relative error    = 8.614069495078867e-16
  momentum relative error= 2.3144484159510705e-05
Gaussian l=1
  grid relative error    = 3.8577061644055427e-16
  momentum relative error= 1.935592451046311e-06


## Arbitrary numerical Legendre examples

In [4]:
solver = legendre_solver(0)

numerical_profiles = [
    ("damped oscillation", lambda r: r * np.exp(-0.35 * r) * np.cos(1.5 * r)),
    ("double bump", lambda r: np.exp(-0.7 * (r - 2.0) ** 2) - 0.6 * np.exp(-0.4 * (r - 6.0) ** 2)),
]

fig, axes = plt.subplots(len(numerical_profiles), 2, figsize=(12, 6.5))
grid_r = np.asarray(solver.transforms.grid_r)
momenta = np.asarray(solver.transforms.momenta)

for row, (name, profile) in enumerate(numerical_profiles):
    coeffs = solver.from_grid_vector(lambda r: jnp.asarray(profile(np.asarray(r))))
    reconstructed = np.asarray(solver.to_grid_vector(coeffs))
    transformed = np.asarray(solver.fourier(coeffs))
    expected = profile(grid_r)
    print(name)
    print("  grid relative error =", relative_error(reconstructed, expected))
    print("  first five k-space samples =", transformed[:5])
    axes[row, 0].plot(grid_r, expected, label="input profile", linewidth=2.0)
    axes[row, 0].plot(grid_r, reconstructed, "--", label="mesh reconstruction", linewidth=1.8)
    axes[row, 0].set_title(f"{name} in r-space")
    axes[row, 0].set_xlabel("r")
    axes[row, 0].set_ylabel("u(r)")
    axes[row, 0].legend()
    axes[row, 1].plot(momenta, transformed, linewidth=2.0)
    axes[row, 1].set_title(f"{name} in k-space")
    axes[row, 1].set_xlabel("k")
    axes[row, 1].set_ylabel(r"$\tilde u(k)$")

fig.tight_layout()


damped oscillation
  grid relative error = 8.119852556163831e-16
  first five k-space samples = [-0.28439337 -0.29318391 -0.3072713  -0.31188557 -0.31355335]
double bump
  grid relative error = 8.557422004888275e-16
  first five k-space samples = [0.33348831 0.40181663 0.58744782 0.83866039 1.08797288]


## Hydrogen Laguerre example

In [5]:
def hydrogen_solver(l: int) -> lm.boundary._types.Solver:
    return lm.compile(
        mesh=lm.MeshSpec("laguerre", "x", n=30, scale=2.0),
        channels=(lm.ChannelSpec(l=l, threshold=0.0, mass_factor=0.5),),
        operators=("T", "1/r"),
        solvers=("spectrum", "wavefunction"),
        grid=jnp.linspace(0.0, 40.0, 4000),
        momenta=jnp.linspace(0.0, 6.0, 600),
    )


def hydrogen_potential(solver: lm.boundary._types.Solver) -> jnp.ndarray:
    return jnp.asarray((-1.0 / solver.mesh.radii)[None, None, :])


solver = hydrogen_solver(0)
spectrum = solver.spectrum(hydrogen_potential(solver))
coeffs_1s = spectrum.eigenvectors[:, 0]
u_r = np.asarray(solver.to_grid_vector(coeffs_1s))
u_k = np.asarray(solver.fourier(coeffs_1s))
print("Hydrogen 1s energy:", float(np.asarray(spectrum.eigenvalues)[0]) * solver.channels[0].mass_factor)
print("First five r-space samples:", u_r[:5])
print("First five k-space samples:", u_k[:5])


Hydrogen 1s energy: -0.50000000002896
First five r-space samples: [0.         0.0198059  0.03921756 0.05824086 0.07688162]
First five k-space samples: [1.59573468 1.5955999  1.59513568 1.59432566 1.59321283]
